In [1]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from tools import DuckDuckGoSearchRun, weather_info_tool, hub_stats_tool
from retriever import guest_info_tool

1 week ago - The officeholder is also honorary proto-canon of the Archbasilica of Saint John Lateran in Rome, although some have rejected the title in the past. The current president is Emmanuel Macron, who succeeded François Hollande on 14 May 2017 following the 2017 presidential election, and was inaugurated ... 1 day ago - Emmanuel Jean-Michel Frédéric Macron (born 21 December 1977) is a French politician who has served as President of France and Co-Prince of Andorra since 2017. He served as Minister of Economics and Finance under President François Hollande from 2014 to 2016. 20 hours ago - A 1962 referendum held under the ... vote. Since then, ten presidential elections have taken place. The 25th and current officeholder has been Emmanuel Macron since 14 May 2017.... 3 weeks ago - Emmanuel Jean-Michel Frédéric Macron CBE (French: [emanɥɛl makʁɔ̃]; born 21 December 1977 in Amiens) is a French politician, senior civil servant, and former investment banker. 1 day ago - François Holla

In [2]:
# Initialize the web search tool
search_tool = DuckDuckGoSearchRun()

import os
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# Generate the chat interface, including the tools
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    huggingfacehub_api_token=HF_TOKEN,
)

chat = ChatHuggingFace(llm=llm, verbose=True)
tools = [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool]
chat_with_tools = chat.bind_tools(tools)

# Generate the AgentState and Agent graph
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])],
    }

## The graph
builder = StateGraph(AgentState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message requires a tool, route to tools
    # Otherwise, provide a direct response
    tools_condition,
)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

In [3]:
response = alfred.invoke({"messages": "Tell me about 'Lady Ada Lovelace'"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
Lady Ada Lovelace, born Augusta Ada King, Countess of Lovelace (10 December 1815 – 27 November 1852), was an English mathematician and writer. She is widely recognized for her work on Charles Babbage's proposed mechanical general-purpose computer, the Analytical Engine. In 1843, she published what is considered the first algorithm intended to be processed by a machine, earning her the title "the first computer programmer."

Key points about Ada Lovelace include:

1. **Early Life**: She was the daughter of the famous poet Lord Byron and showed early interest in mathematics and science.
2. **Meeting Charles Babbage**: At age 17, she met Babbage and became fascinated by his work on the Analytical Engine.
3. **Contributions**: She wrote detailed notes on the Analytical Engine, including an algorithm for computing Bernoulli numbers, which is seen as the first computer program.
4. **Legacy**: Her contributions have gained recognition over time, and she is celebrated as a

In [4]:
response = alfred.invoke({"messages": "What's the weather like in Paris tonight? Will it be suitable for our fireworks display?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
The weather in Paris tonight is windy with a temperature of 20°C. While the temperature is mild, the windy conditions might not be ideal for a fireworks display, as wind can affect the trajectory and safety of fireworks. It would be best to monitor the forecast closer to the time to ensure optimal conditions.


In [5]:
response = alfred.invoke({"messages": "One of our guests is from Qwen. What can you tell me about their most popular model?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
The most popular model by Qwen is Qwen/Qwen3-0.6B, which has garnered an impressive number of downloads, totaling 21,156,913. This highlights its significance and widespread use within the community.


In [10]:
response = alfred.invoke({"messages":"I need to speak with 'Dr. Nikola Tesla' about recent advancements in wireless energy. Can you help me prepare for this conversation?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
You can reach out to Dr. Nikola Tesla at **nikola.tesla@gmail.com**. Since he’s passionate about pigeons, you could start the conversation by mentioning something about them—perhaps sharing an anecdote or asking about his latest pigeon-related observations. 

Additionally, since he recently patented a new wireless energy transmission system, you might want to prepare a few questions about his advancements, such as:

1. What inspired the latest breakthrough in wireless energy transmission?
2. How does this new system differ from previous technologies?
3. Are there any practical applications for this technology in everyday life?

Let me know if you'd like help drafting an email or preparing more detailed questions!


In [11]:
# First interaction
response = alfred.invoke({"messages": [HumanMessage(content="Tell me about 'Lady Ada Lovelace'. What's her background and how is she related to me?")]})


print("🎩 Alfred's Response:")
print(response['messages'][-1].content)
print()

# Second interaction (referencing the first)
response = alfred.invoke({"messages": response["messages"] + [HumanMessage(content="What projects is she currently working on?")]})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
Lady Ada Lovelace is your best friend, and she is an esteemed mathematician known for her pioneering contributions to mathematics and computing. She is often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine. Her expertise and legacy in the field of early computing make her a remarkable figure in history.

As for how she is related to you, she is your best friend, which means you share a close and personal bond. If you ever need assistance or want to connect with her, you can reach out to her at ada.lovelace@example.com.

🎩 Alfred's Response:
I don't have real-time updates on Lady Ada Lovelace's current projects since she lived in the 19th century. However, historically, she is most notably recognized for her work on Charles Babbage's Analytical Engine, where she wrote what is considered the first algorithm intended for processing by a machine. If you're interested in her historical contributions or want to explore m